# 00 — Exploratory Data Analysis: FRED-MD

This notebook performs EDA on the FRED-MD macroeconomic dataset before model training.
All analysis uses **training data only** to avoid data leakage.

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

from tcc_itransformer.config import ExperimentConfig
from tcc_itransformer.data.fred_md import load_fred_md
from tcc_itransformer.data.preprocessing import FredMDPreprocessor
from tcc_itransformer.utils.viz import (
    plot_correlation_heatmaps,
    plot_dist_histograms_grid,
    plot_missing_data_heatmap,
    plot_stationarity_summary,
    plot_window_statistics,
)

warnings.filterwarnings("ignore", category=FutureWarning)

config = ExperimentConfig.from_yaml("../configs/default.yaml")
FIGURES_DIR = Path("../results/figures/eda")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Config: W={config.window_size}, d_model={config.d_model}, latent={config.latent_dim}")

## 1. Load FRED-MD Data

In [ ]:
df_raw = load_fred_md(config.data_path)
print(f"Shape: {df_raw.shape}")
print(f"Date range: {df_raw.index.min()} to {df_raw.index.max()}")
print(f"Series: {df_raw.columns.tolist()[:10]} ... ({len(df_raw.columns)} total)")
df_raw.head()

## 2. Missing Data Analysis

In [ ]:
null_pct = df_raw.isnull().mean().sort_values(ascending=False)
print(f"Series with >10% missing: {(null_pct > 0.1).sum()}")
print(f"Series with any missing: {(null_pct > 0).sum()}")
print(f"\nTop 10 missing:")
display(null_pct.head(10).to_frame("pct_missing"))

fig = plot_missing_data_heatmap(df_raw, save_path=FIGURES_DIR / "missing_data.png")
plt.show()

## 3. Train/Val/Test Split

In [ ]:
train_mask = df_raw.index <= config.train_end
val_mask = (df_raw.index > config.train_end) & (df_raw.index <= config.val_end)
test_mask = df_raw.index > config.val_end

df_train = df_raw[train_mask]
df_val = df_raw[val_mask]
df_test = df_raw[test_mask]

print(f"Train: {len(df_train)} rows ({df_train.index.min()} to {df_train.index.max()})")
print(f"Val:   {len(df_val)} rows ({df_val.index.min()} to {df_val.index.max()})")
print(f"Test:  {len(df_test)} rows ({df_test.index.min()} to {df_test.index.max()})")

## 4. Distributions (Train Only)

In [ ]:
# Top 12 series by kurtosis
kurt = df_train.kurtosis().sort_values(ascending=False)
top_12 = kurt.head(12).index.tolist()
print("Top 12 by kurtosis:")
display(kurt.head(12).to_frame("kurtosis"))

fig = plot_dist_histograms_grid(df_train, columns=top_12, save_path=FIGURES_DIR / "distributions.png")
plt.show()

In [ ]:
# Jarque-Bera normality test for all series
jb_results = []
for col in df_train.columns:
    data = df_train[col].dropna()
    if len(data) > 10:
        stat, p = stats.jarque_bera(data)
        jb_results.append({"series": col, "jb_stat": stat, "p_value": p, "normal": p > 0.05})

jb_df = pd.DataFrame(jb_results).sort_values("p_value", ascending=False)
print(f"Normally distributed (p>0.05): {jb_df['normal'].sum()}/{len(jb_df)}")
jb_df.head(10)

## 5. Stationarity (ADF + KPSS)

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss

adf_pvals, kpss_pvals, names = [], [], []
for col in df_train.columns:
    data = df_train[col].dropna()
    if len(data) > 20:
        try:
            adf_p = adfuller(data, autolag="AIC")[1]
            kpss_p = kpss(data, regression="c", nlags="auto")[1]
            adf_pvals.append(adf_p)
            kpss_pvals.append(kpss_p)
            names.append(col)
        except Exception:
            pass

adf_arr = np.array(adf_pvals)
kpss_arr = np.array(kpss_pvals)
print(f"Stationary by ADF (p<0.05): {(adf_arr < 0.05).sum()}/{len(adf_arr)}")
print(f"Stationary by KPSS (p>0.05): {(kpss_arr > 0.05).sum()}/{len(kpss_arr)}")

fig = plot_stationarity_summary(adf_arr, kpss_arr, names, save_path=FIGURES_DIR / "stationarity.png")
plt.show()

## 6. Split Visualization + KS Test

In [ ]:
# KS test: train vs test distribution shift per series
ks_results = []
for col in df_train.columns:
    tr = df_train[col].dropna()
    te = df_test[col].dropna()
    if len(tr) > 10 and len(te) > 5:
        stat, p = stats.ks_2samp(tr, te)
        ks_results.append({"series": col, "ks_stat": stat, "p_value": p, "shifted": p < 0.05})

ks_df = pd.DataFrame(ks_results).sort_values("p_value")
print(f"Distribution shift detected (p<0.05): {ks_df['shifted'].sum()}/{len(ks_df)}")
ks_df.head(10)

## 7. Correlation Structure

In [ ]:
fig = plot_correlation_heatmaps(df_train.dropna(axis=1), save_path=FIGURES_DIR / "correlation.png")
plt.show()

## 8. Window Statistics

In [ ]:
# Compute per-window mean and std for W=12
W = config.window_size
train_vals = df_train.select_dtypes(include=[np.number]).dropna(axis=1).values
n_windows = len(train_vals) - W + 1
w_means = np.array([train_vals[i:i+W].mean() for i in range(n_windows)])
w_stds = np.array([train_vals[i:i+W].std() for i in range(n_windows)])

print(f"Number of windows (W={W}): {n_windows}")
fig = plot_window_statistics(w_means, w_stds, save_path=FIGURES_DIR / "window_stats.png")
plt.show()

## 9. Baseline PCA on Raw Features

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from tcc_itransformer.utils.viz import plot_scree

clean_train = df_train.select_dtypes(include=[np.number]).dropna(axis=1)
X_scaled = StandardScaler().fit_transform(clean_train)

pca = PCA(n_components=min(10, X_scaled.shape[1]))
pca.fit(X_scaled)

cumvar = np.cumsum(pca.explained_variance_ratio_)
n_90 = np.searchsorted(cumvar, 0.9) + 1
print(f"Components for 90% variance: {n_90}")
print(f"Explained variance (first 5): {pca.explained_variance_ratio_[:5].round(4)}")

fig = plot_scree(pca, save_path=FIGURES_DIR / "baseline_pca_scree.png")
plt.show()

## 10. Summary

| Item | Result |
|------|--------|
| Total series | TBD |
| Series excluded (>10% NaN) | TBD |
| Train samples | TBD |
| Normally distributed series | TBD |
| Stationary series (ADF) | TBD |
| Distribution shift (train→test) | TBD |
| PCA components for 90% var (raw) | TBD |

*Fill in after running all cells.*